# Reto 04 — Pandas I: cargar y limpiar

**Caso:** heredaste `ventas.csv` de la tienda. Nadie sabe quien lo genero y esta sucio.
Antes de que alguien lo use para decidir algo, tienes que dejarlo confiable.

Tu entregable es un `ventas_limpio.csv` **y la lista de decisiones que tomaste**.
Limpiar datos no es apretar botones: cada nulo que rellenas o cada fila que borras
es una decision que alguien te va a cuestionar en una reunion.

In [ ]:
import pandas as pd

RUTA = "../../_RECURSOS/datasets/ventas.csv"

## 1 — Cargar bien

El archivo NO usa coma como separador ni punto como decimal. Si `df.shape` te da
una sola columna, o si `precio_unitario` sale como `object`, la carga esta mal.

In [ ]:
# TODO: carga el archivo con los parametros correctos
ventas = ...


# --- verificacion (no modificar) ---
assert ventas.shape == (491, 10), f"Esperado (491, 10), salio {ventas.shape}"
assert ventas["precio_unitario"].dtype.kind == "f", "precio_unitario debe ser numerico, no texto"
print("OK 1 — cargado")
ventas.head()

## 2 — Radiografia

Antes de tocar nada, mira. Responde en la celda markdown de abajo, con numeros:

- Cuantas filas y columnas hay
- Que tipo tiene cada columna y cual esta mal tipada
- Cuantos nulos hay por columna
- Cuantas filas duplicadas exactas
- Cuantos valores distintos tiene `canal` y por que son mas de 3

In [ ]:
# TODO: info, describe, isna().sum(), duplicated().sum(), value_counts()

### Tus hallazgos

*(escribe aqui, con numeros)*

1. ...
2. ...

## 3 — Duplicados

Elimina las filas duplicadas **exactas**. Guarda en `ventas_sin_dup`.

Pregunta que si se evalua: por que borrar duplicados exactos es seguro aqui, pero
NO lo seria si `id_venta` se repitiera con importes distintos?

In [ ]:
ventas_sin_dup = ...


# --- verificacion (no modificar) ---
assert len(ventas_sin_dup) == 482, f"Esperado 482, salio {len(ventas_sin_dup)}"
print("OK 3 — 491 -> 482")

## 4 — Normalizar `canal`

`canal` tiene 12 variantes de 3 valores reales (` Web `, `WEB`, `web`...).
Dejalo en exactamente 3: `Web`, `Tienda`, `Telefono`.

Pista: el accesor `.str` encadena. `.str.strip().str.title()`.

In [ ]:
# TODO


# --- verificacion (no modificar) ---
assert sorted(ventas_sin_dup["canal"].unique()) == ["Telefono", "Tienda", "Web"], \
    f"Salio {sorted(ventas_sin_dup['canal'].unique())}"
print("OK 4 — 3 canales")

## 5 — Fechas

`fecha` es texto y viene en dos formatos: `2025-03-17` y `17/03/2025`.
Conviertela a `datetime` **sin perder ninguna fila**.

Pista: `pd.to_datetime(..., format="mixed", dayfirst=True)`.

Trampa: si usas `errors="coerce"` sin mirar, conviertes los fallos en `NaT` en silencio.
Comprueba despues cuantos `NaT` quedaron.

In [ ]:
# TODO


# --- verificacion (no modificar) ---
assert ventas_sin_dup["fecha"].dtype.kind == "M", "fecha debe ser datetime64"
assert ventas_sin_dup["fecha"].isna().sum() == 0, "se perdieron fechas al convertir"
assert ventas_sin_dup["fecha"].dt.year.unique().tolist() == [2025]
print("OK 5 — fechas convertidas, 0 perdidas")

## 6 — Nulos: dos columnas, dos decisiones distintas

**`importe`** (12 nulos): NO lo rellenes con la media. Lo puedes **recalcular** exacto:
`cantidad * precio_unitario * (1 - descuento_pct/100)`.

**`ciudad`** (19 nulos): no se puede deducir. Tienes tres opciones —borrar la fila,
poner `"Desconocida"`, o imputar la ciudad mas frecuente del cliente. Elige una,
**hazla, y justificala por escrito**. Cualquiera de las tres es correcta si esta argumentada.

In [ ]:
# TODO: recalcular importe


# --- verificacion (no modificar) ---
assert ventas_sin_dup["importe"].isna().sum() == 0, "quedan importes nulos"
print("OK 6a — importe completo")

In [ ]:
# TODO: tu decision sobre ciudad

### Tu decision sobre `ciudad`

*(que hiciste y por que — 2 o 3 lineas)*

## 7 — El dato raro

Hay una venta con `cantidad` negativa. Encuentrala, decide si es un error de carga o
una devolucion legitima, y documenta que hiciste con ella.

No hay respuesta unica. Hay respuestas justificadas y respuestas silenciosas.
Las silenciosas no valen.

In [ ]:
# TODO

### Tu decision sobre la cantidad negativa

*(...)*

## 8 — Primeras respuestas de negocio

Con el DataFrame limpio, responde:

1. Cual es el importe total facturado en 2025?
2. Cual es la ciudad que mas factura?
3. Cual es el ticket promedio por canal?
4. En que mes se facturo mas?

In [ ]:
# TODO

## 9 — Exportar

Guarda el resultado como `ventas_limpio.csv` **al lado de este notebook**, con `sep=";"`,
`decimal=","` y **sin la columna de indice**.

In [ ]:
# TODO


# --- verificacion (no modificar) ---
import os
assert os.path.exists("ventas_limpio.csv"), "no se genero el archivo"
control = pd.read_csv("ventas_limpio.csv", sep=";", decimal=",")
assert "Unnamed: 0" not in control.columns, "exportaste el indice; usa index=False"
assert control["importe"].isna().sum() == 0
print(f"OK 9 — ventas_limpio.csv con {len(control)} filas")

---
## Criterios de aceptacion

Antes de entregar, verifica que se cumpla **todo** lo de abajo.

- [ ] *Restart & Run All* corre limpio y todos los `OK` salen
- [ ] Se genera `ventas_limpio.csv` con `sep=';'`, `decimal=','` e `index=False`
- [ ] `canal` tiene exactamente 3 valores
- [ ] `fecha` es `datetime64` y no hay ni un `NaT`
- [ ] `importe` no tiene nulos y esta **recalculado**, no imputado con la media
- [ ] Las 3 celdas markdown de decision (`ciudad`, cantidad negativa, hallazgos) estan escritas, no vacias
- [ ] Las rutas son relativas: si aparece `C:\Users\...` es −4